# build 

In [22]:
from pydantic import BaseModel, Field, EmailStr
from typing import Optional, Literal
from pydantic_ai import Agent
from constance import MODEL_SMALL, MODEL_MEDIUM, MODEL_LARGE
from dotenv import load_dotenv

load_dotenv()


class EmailExtractor(BaseModel):
    sender_name: Optional[str]
    sender_email: EmailStr
    issue_catagory: Literal["billing", "damaged product", "technical", "other"]
    urgency: Literal["low", "medium", "high"]
    summay: str = Field(desceiption="summarize email in two sentences.")


email_extractor_agent = Agent(
    MODEL_LARGE,
    system_prompt=""" your are customer support agent, your task is to extract relevant informaiton form an email """,
    output_type=EmailExtractor,
)

C:\Users\alici\AppData\Local\Temp\ipykernel_4984\3068786584.py:15: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'desceiption'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  summay: str = Field(desceiption="summarize email in two sentences.")


In [23]:
import pandas as pd

df = pd.read_json("cleanded_email")
df

,inputs,expectations
0,{'email': 'From: Erik Lindqvist <erik.lindqvis...,"{'expected_response': '{""sender_name"": ""Erik L..."
1,{'email': 'From: Maja Bergström <maja.bergstro...,"{'expected_response': '{""sender_name"": ""Maja B..."
2,{'email': 'From: Oscar Johansson <oscar.johans...,"{'expected_response': '{""sender_name"": ""Oscar ..."
3,{'email': 'From: Linnea Karlsson <linnea.karls...,"{'expected_response': '{""sender_name"": ""Linnea..."


In [24]:
sample_mail = df.iloc[2]["inputs"]["email"]
sample_mail


"From: Oscar Johansson <oscar.johansson@yahoo.se>\nSubject: Cannot access my account for 3 days - Urgent help needed\n\nHello Support Team,\n\nI am reaching out because I have been completely locked out of my account for the past three days and I am running out of ideas on how to fix this on my own. The problem started on Monday evening when I tried to log in as usual but kept receiving an 'Invalid credentials' error despite being absolutely certain that I was entering the correct password.\n\nI followed the instructions on your website to reset my password, but the problem is that the password reset email never arrives in my inbox. I have checked my spam and junk folders multiple times, and there is nothing there either. I have attempted the reset process at least six or seven times across different browsers and even from my phone, but the result is always the same - no email arrives.\n\nThis is causing me real problems because I have important documents and data stored in my account 

In [25]:
result = await email_extractor_agent.run(sample_mail)
result


AgentRunResult(output=EmailExtractor(sender_name='Oscar Johansson', sender_email='oscar.johansson@yahoo.se', issue_catagory='technical', urgency='high', summay='Oscar Johansson has been locked out of his account for three days due to invalid credentials errors. He has attempted multiple password reset procedures but the reset emails are not arriving in his inbox or spam folder, preventing him from accessing important work documents with an upcoming Friday deadline.'))

In [26]:
result.output

EmailExtractor(sender_name='Oscar Johansson', sender_email='oscar.johansson@yahoo.se', issue_catagory='technical', urgency='high', summay='Oscar Johansson has been locked out of his account for three days due to invalid credentials errors. He has attempted multiple password reset procedures but the reset emails are not arriving in his inbox or spam folder, preventing him from accessing important work documents with an upcoming Friday deadline.')

In [27]:
from mlflow.genai import load_prompt



class EmailExtractor(BaseModel):
    sender_name: Optional[str]
    sender_email: EmailStr
    issue_catagory: Literal["billing", "damaged product", "technical", "other"]
    urgency: Literal["low", "medium", "high"] =  Field(description= load_prompt("email-urgency-description").format())
    summay: str = Field(desceiption= load_prompt("summay_description").format(num_sentences = 4))


email_extractor_agent = Agent(
    MODEL_LARGE,
    system_prompt=load_prompt("email-extractor-system-prompt").format(),
    output_type=EmailExtractor,
)

C:\Users\alici\AppData\Local\Temp\ipykernel_4984\2357203225.py:10: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'desceiption'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  summay: str = Field(desceiption= load_prompt("summay_description").format(num_sentences = 4))


In [28]:
result = await email_extractor_agent.run(sample_mail)
result.output

EmailExtractor(sender_name='Oscar Johansson', sender_email='oscar.johansson@yahoo.se', issue_catagory='technical', urgency='high', summay='Oscar Johansson has been locked out of his account for three days due to invalid credentials errors. He has attempted to reset his password multiple times but the reset emails are not arriving in his inbox, spam, or junk folders. This is preventing him from accessing important work documents and data with a deadline approaching this Friday. He has tried various troubleshooting steps including different browsers, incognito mode, and clearing cache without success.')

In [29]:
result.output.model_dump()

{'sender_name': 'Oscar Johansson',
 'sender_email': 'oscar.johansson@yahoo.se',
 'issue_catagory': 'technical',
 'urgency': 'high',
 'summay': 'Oscar Johansson has been locked out of his account for three days due to invalid credentials errors. He has attempted to reset his password multiple times but the reset emails are not arriving in his inbox, spam, or junk folders. This is preventing him from accessing important work documents and data with a deadline approaching this Friday. He has tried various troubleshooting steps including different browsers, incognito mode, and clearing cache without success.'}

In [30]:
df

,inputs,expectations
0,{'email': 'From: Erik Lindqvist <erik.lindqvis...,"{'expected_response': '{""sender_name"": ""Erik L..."
1,{'email': 'From: Maja Bergström <maja.bergstro...,"{'expected_response': '{""sender_name"": ""Maja B..."
2,{'email': 'From: Oscar Johansson <oscar.johans...,"{'expected_response': '{""sender_name"": ""Oscar ..."
3,{'email': 'From: Linnea Karlsson <linnea.karls...,"{'expected_response': '{""sender_name"": ""Linnea..."


In [31]:
df["outputs"] = [{},{},result.output.model_dump(), {}]
df

,inputs,expectations,outputs
0,{'email': 'From: Erik Lindqvist <erik.lindqvis...,"{'expected_response': '{""sender_name"": ""Erik L...",{}
1,{'email': 'From: Maja Bergström <maja.bergstro...,"{'expected_response': '{""sender_name"": ""Maja B...",{}
2,{'email': 'From: Oscar Johansson <oscar.johans...,"{'expected_response': '{""sender_name"": ""Oscar ...","{'sender_name': 'Oscar Johansson', 'sender_ema..."
3,{'email': 'From: Linnea Karlsson <linnea.karls...,"{'expected_response': '{""sender_name"": ""Linnea...",{}


In [32]:
df_sample = df.drop([0,1,3])
df_sample

,inputs,expectations,outputs
2,{'email': 'From: Oscar Johansson <oscar.johans...,"{'expected_response': '{""sender_name"": ""Oscar ...","{'sender_name': 'Oscar Johansson', 'sender_ema..."


# judge

In [33]:
from mlflow.genai.scorers import Correctness, Summarization, Completeness, Fluency
import mlflow

llm_judge = "openrouter:/nvidia/nemotron-3-super-120b-a12b:free"

with mlflow.start_run(run_name="email-extractor-evaluator"):
    mlflow.log_param("model_ju", llm_judge)
    mlflow.log_param("model", MODEL_LARGE)
    results = mlflow.genai.evaluate(
        data = df_sample,
        scorers=[Correctness(model=llm_judge),
                 Summarization(model=llm_judge)]
        )
      
results   

#get_all_scorers()

Evaluating:   0%|          | 0/1 [Elapsed: 00:00, Remaining: ?]


✨ Evaluation completed.

Metrics and evaluation results are logged to the MLflow run:
  Run name: email-extractor-evaluator
  Run ID: 35e82cfd94b8424abf36e8718fcddc67

To view the detailed evaluation results with sample-wise scores,
open the Traces tab in the Run page in the MLflow UI.



EvaluationResult(
  run_id: 35e82cfd94b8424abf36e8718fcddc67
  metrics:
    correctness/mean: 1.0
    summarization/mean: 1.0
  result_df: 1 rows x 15 cols
)

In [34]:
results.result_df

,trace_id,correctness/value,summarization/value,expected_response/value,trace,client_request_id,state,request_time,execution_duration,request,response,trace_metadata,tags,spans,assessments
0,tr-f58d6d8393d86d0432a9458245d37738,yes,yes,"{""sender_name"": ""Oscar Johansson"", ""sender_ema...","{""info"": {""trace_id"": ""tr-f58d6d8393d86d0432a9...",None,OK,1776258678653,0,{'email': 'From: Oscar Johansson <oscar.johans...,"{'sender_name': 'Oscar Johansson', 'sender_ema...","{'mlflow.user': 'alici', 'mlflow.source.name':...",{'mlflow.artifactLocation': 'file:c:/Users/ali...,"[{'trace_id': '9Y1tg5PYbQQyqUWCRdN3OA==', 'spa...",[{'assessment_id': 'a-f77852c73caa4df5b6e9030c...
